In [1]:
# Celda M1 - Configuracion y rutas
import io, json, re, ast, time
import numpy as np
import pandas as pd
import zstandard as zstd
from pathlib import Path
from collections import defaultdict, Counter
from datetime import date, timedelta

import os
BASE   = Path(os.environ.get('TESIS_BASE', '/Users/ppizam/Claude/Master Thesis'))
DATA   = Path(os.environ.get('REDDIT_DATA', '/Users/ppizam/Library/CloudStorage/GoogleDrive-pizacapital@gmail.com/Other computers/My Mac RRG/data/reddit'))
MATRIX = BASE / 'Desarrollo/Metodologia/Matrix'
PADRON = BASE / 'Desarrollo/Metodologia/Lista Maestra de Tickers/Lista maestra V2/padron_vigencias_2020_2026_ver03_1.csv'
SCRIPT_VIEJO = BASE / 'Phyton Tesis/analisis_menciones_reddit.py'

(MATRIX / 'mensuales').mkdir(exist_ok=True)
(MATRIX / 'maestras').mkdir(exist_ok=True)
(MATRIX / 'logs').mkdir(exist_ok=True)

for ruta in (DATA, PADRON, SCRIPT_VIEJO):
    assert ruta.exists(), f'no se encuentra: {ruta}'
print('rutas OK')

rutas OK


In [24]:
# Celda M2 v2 - Universo de deteccion (cierra el hueco de closed-end funds)
pad = pd.read_csv(PADRON, keep_default_na=False, na_values=[''],
                  parse_dates=['fecha_inicio', 'fecha_fin'])
assert len(pad) == 13900
assert (pad['ticker'] == 'NA').sum() == 2

shrcd = pd.to_numeric(pad['shrcd'], errors='coerce')
es_no_accion = (
    shrcd.notna() & (
        (shrcd // 10).isin([7, 2, 4])      # fondos/units, certificados, SBI
        | (shrcd % 10).isin([4, 5])        # closed-end funds (14, 15...): el caso CHN
    )
)
uni = pad[~es_no_accion].copy()
print(len(pad), 'vidas ->', len(uni), 'vidas de accion en el universo')
print('tickers distintos:', uni['ticker'].nunique())
# Esperado: unas decenas menos que la corrida anterior (los CEF fuera)

13900 vidas -> 9309 vidas de accion en el universo
tickers distintos: 9034


In [66]:
# Celda M3 v11 - Buscador CONGELADO (8 pasadas del piloto + 2 pasadas de validacion anual 2021-2026)
arbol = ast.parse(SCRIPT_VIEJO.read_text())
listas = {}
for nodo in ast.walk(arbol):
    if isinstance(nodo, ast.Assign):
        for obj in nodo.targets:
            if isinstance(obj, ast.Name) and obj.id in ('SIMBOLOS_EXCLUIDOS', 'PALABRAS_COMUNES_EN'):
                listas[obj.id] = set(ast.literal_eval(nodo.value))
SIMBOLOS_EXCLUIDOS = listas['SIMBOLOS_EXCLUIDOS']
PALABRAS_COMUNES_EN = listas['PALABRAS_COMUNES_EN']

JERGA = {'DTE', 'BLSH', 'BULL', 'BEAR', 'RSI', 'NDAQ', 'IBKR', 'YOLO', 'FOMO',
         'HODL', 'ATH', 'ITM', 'OTM', 'THETA', 'GAMMA', 'DELTA', 'VEGA',
         'CALLS', 'PUTS', 'STONK', 'TENDIES', 'MOASS', 'DRS', 'NFA', 'DYOR',
         'FOMC', 'CPI', 'EOD', 'HOLD', 'BUY', 'SELL',
         # novena pasada (validacion anual 2021-2026):
         'LFG', 'MSM', 'LINK', 'TACO',
         # decima pasada (segunda capa de la validacion anual):
         'GET', 'DTC', 'LMAO'}
INSTITUCIONES = {'IRS', 'SEC', 'FDA', 'FBI', 'CIA', 'DOJ', 'EPA', 'CDC'}
SIMBOLOS_EXCLUIDOS = SIMBOLOS_EXCLUIDOS | JERGA | INSTITUCIONES

EXTRA_COMUNES = {'com', 'china', 'best', 'guess', 'gold', 'data', 'max', 'job',
                 'jobs', 'solar', 'silver', 'coffee', 'sugar', 'water', 'games',
                 'chart', 'reading', 'experience', 'crush', 'america', 'american',
                 'fidelity', 'supply', 'investors',
                 'community', 'store', 'mobile', 'switch',
                 'recovery', 'nasdaq',
                 'truck', 'liquid', 'orange', 'california', 'powell', 'safety',
                 'concept',
                 'golden', 'lifetime', 'formula', 'micro',
                 'select', 'progress',
                 # novena pasada (validacion anual 2021-2026):
                 'growth', 'online', 'states', 'currency', 'asset', 'matters',
                 'learn', 'crypto', 'perfect', 'bullish', 'absolute',
                 'advantage', 'parts', 'cohen', 'appreciate', 'decent',
                 'israel', 'infinity', 'ladder', 'belong', 'metals', 'copper',
                 'distribution', 'infrastructure', 'defense',
                 # decima pasada (segunda capa de la validacion anual):
                 'infinite', 'driven', 'professional', 'buckle'}
PALABRAS_COMUNES_EN = PALABRAS_COMUNES_EN | EXTRA_COMUNES

STOP_CORP = {'inc', 'incorporated', 'corp', 'corporation', 'ltd', 'limited', 'plc',
             'co', 'company', 'companies', 'holdings', 'holding', 'group', 'class',
             'common', 'stock', 'shares', 'share', 'adr', 'adrs', 'ordinary',
             'trust', 'fund', 'sa', 'nv', 'ag', 'spa', 'the', 'of', 'de', 'and',
             'new', 'del', 'international', 'technologies', 'technology', 'tech',
             'investment', 'investments', 'capital', 'financial', 'finance',
             'industries', 'industrial', 'resources', 'systems', 'solutions',
             'partners', 'properties', 'brands', 'media', 'digital', 'energy',
             'enterprises', 'ventures', 'labs', 'pharmaceuticals', 'pharma',
             'bancorp', 'bancshares', 'bank', 'services',
             'equity', 'liquidity', 'research', 'markets', 'trading',
             'strategies', 'strategy', 'acquisition', 'acquisitions',
             'insurance', 'customers', 'consumer', 'products', 'national',
             'global', 'western', 'southern', 'northern', 'pacific',
             'premier', 'worldwide', 'standard',
             'security', 'healthcare', 'mortgage', 'diversified', 'education',
             'payments', 'network', 'citizens', 'institutions',
             'entertainment', 'software', 'airlines', 'banking', 'sports',
             'equipment', 'drilling', 'homes', 'home', 'merger', 'mergers',
             'mining',
             'federal', 'independent', 'associated', 'republic', 'peoples',
             'heritage', 'pioneer', 'liberty', 'enterprise'}

FRASES_EXCLUIDAS = {'real estate', 'united states',
                    # decima pasada (segunda capa de la validacion anual):
                    'very good'}

MIN_KW = 5

cashtag_de = {}
token_de = {}
kw1_de = defaultdict(set)
kwfrase = defaultdict(list)

for t, nombre in uni[['ticker', 'comnam']].drop_duplicates().itertuples(index=False):
    T = str(t).upper()
    cashtag_de[T] = t
    if 3 <= len(T) <= 5 and T.isalpha() and T not in SIMBOLOS_EXCLUIDOS:
        token_de[T] = t
    limpio = re.sub(r'[^a-z ]', ' ', str(nombre).lower())
    palabras = [w for w in limpio.split() if len(w) >= 3 and w not in STOP_CORP]
    if len(palabras) == 1:
        w = palabras[0]
        if len(w) >= MIN_KW and w not in PALABRAS_COMUNES_EN:
            kw1_de[w].add(t)
    elif len(palabras) >= 2:
        frase = ' '.join(palabras[:2])
        if frase not in FRASES_EXCLUIDAS:
            kwfrase[palabras[0]].append((frase, t))

(MATRIX / 'logs/exclusiones_congeladas.txt').write_text(
    '\n'.join(sorted(SIMBOLOS_EXCLUIDOS)) + '\n---PALABRAS---\n' +
    '\n'.join(sorted(PALABRAS_COMUNES_EN)) + '\n---FRASES---\n' +
    '\n'.join(sorted(FRASES_EXCLUIDAS)))

print('cashtags:', len(cashtag_de), '| tokens:', len(token_de),
      '| keywords 1 palabra:', len(kw1_de), '| frases:', sum(len(v) for v in kwfrase.values()))

CULPABLES = ['JD', 'WBAI', 'WUBA', 'CHN', 'BEST', 'GES', 'RMAX', 'AGNC', 'SLRC',
             'BTG', 'NGD', 'USAU', 'TECD', 'DAIO', 'JOBS',
             'ISBC', 'GTLS', 'ACMR', 'RDI', 'RDIB', 'EXPC', 'HCR', 'EQBK',
             'LQDT', 'FDBC', 'HDS', 'BAC',
             'KREF', 'TCFC', 'CTBI', 'STOR', 'SITO', 'SWCH',
             'GPX', 'ERII', 'NDAQ', 'INSU', 'CUBI',
             'NSEC', 'SNFCA', 'HCA', 'NHC', 'AMN', 'HCSG', 'MITT', 'SDI',
             'USAK', 'YVR', 'TAL', 'RYB', 'POWL', 'ORAN', 'CRC', 'MSA',
             'SAFT', 'GPN', 'NTIP', 'GBR', 'CIA', 'FISI',
             'GDEN', 'CVV', 'HEES', 'PACD', 'LGIH', 'MHO', 'KBLM', 'LCUT',
             'SSRM', 'FORTY', 'OIIM',
             'HFBL', 'IBTX', 'INDB', 'SLCT', 'AC', 'PRGS',
             # novena pasada (validacion anual 2021-2026):
             'VYGG', 'LDHA', 'KFII', 'GLBE', 'SSBK', 'DCX', 'DAAQ', 'COHN',
             'ABST', 'ADV', 'ID', 'LCW', 'DAOO', 'PERF', 'BLSH', 'SFR',
             'DXST', 'ISRL', 'AIIA', 'DFNS', 'MTAL', 'NEWP', 'DSGR', 'INFI',
             'LADR', 'SMX', 'BLNG',
             # decima pasada (segunda capa de la validacion anual):
             'NFNT', 'DRVN', 'PFHD', 'BKE', 'GET', 'DTC', 'LMAO', 'VGFC']
for t in CULPABLES:
    kws = [w for w, s in kw1_de.items() if t in s]
    frs = [f for lst in kwfrase.values() for f, tt in lst if tt == t]
    marca = kws + [f for f in frs if f in FRASES_EXCLUIDAS]
    if marca:
        print('AVISO: sigue detectable por via generica ->', t, marca)
for t in INSTITUCIONES:
    if t in token_de:
        print('AVISO: acronimo institucional sigue como token ->', t)
print('candado de keywords, frases y acronimos revisado')

viglist = uni.groupby('ticker').apply(
    lambda g: list(zip(g['fecha_inicio'], g['fecha_fin'])), include_groups=False).to_dict()

def vigentes_del_mes(anio, mes):
    d0 = pd.Timestamp(anio, mes, 1)
    d1 = (d0 + pd.offsets.MonthEnd(0))
    dias = pd.date_range(d0, d1, freq='D')
    out = {d.date(): set() for d in dias}
    for t, intervalos in viglist.items():
        for ini, fin in intervalos:
            lo, hi = max(ini, d0), min(fin, d1)
            if lo <= hi:
                for d in pd.date_range(lo, hi, freq='D'):
                    out[d.date()].add(t)
    return out

print('buscador v11 CONGELADO listo')

cashtags: 9034 | tokens: 8618 | keywords 1 palabra: 3586 | frases: 3692
candado de keywords, frases y acronimos revisado
buscador v11 CONGELADO listo


In [26]:
# Celda M4 - Prueba unitaria del buscador (candado antes de tocar datos reales)
RE_CASHTAG = re.compile(r'\$([A-Za-z]{1,5})\b')
RE_TOKEN   = re.compile(r'(?<![A-Za-z$])[A-Z]{3,5}(?![A-Za-z])')
RE_PALABRA = re.compile(r'[a-z]{3,}')

def detectar(texto):
    """Devuelve el SET de tickers mencionados en un texto (sin filtro de vigencia)."""
    if not texto or texto in ('[removed]', '[deleted]'):
        return set()
    hallados = set()
    for m in RE_CASHTAG.findall(texto):
        t = cashtag_de.get(m.upper())
        if t: hallados.add(t)
    for m in RE_TOKEN.findall(texto):
        t = token_de.get(m)
        if t: hallados.add(t)
    palabras = RE_PALABRA.findall(texto.lower())
    pset = set(palabras)
    for w in pset:
        for t in kw1_de.get(w, ()):
            hallados.add(t)
    texto_lo = ' '.join(palabras)
    for w in pset & kwfrase.keys():
        for frase, t in kwfrase[w]:
            if frase in texto_lo:
                hallados.add(t)
    return hallados

pruebas = [
    ('YOLO $gme calls, apple to the moon', {'GME', 'AAPL'}),
    ('Buying GME and AMC before earnings', {'GME', 'AMC'}),
    ('I love my Tesla and some berkshire hathaway', {'TSLA'}),          # BRK por frase
    ('DTE is 0 today, pure theta gang', set()),                          # jerga excluida
    ('$DTE looks cheap', {'DTE'}),                                       # cashtag si cuenta
    ('the star of the show', set()),                                     # palabra comun
]
ok = 0
for texto, esperado in pruebas:
    r = detectar(texto)
    estado = 'OK ' if esperado <= r and (r - esperado == set() or esperado == set()) else 'REVISAR'
    if estado == 'OK ': ok += 1
    print(f'{estado} | "{texto}" -> {sorted(r)}')
print(ok, 'de', len(pruebas))
# La prueba 3 debe traer TSLA y ademas BRK (berkshire hathaway): revisala visualmente

OK  | "YOLO $gme calls, apple to the moon" -> ['AAPL', 'GME']
OK  | "Buying GME and AMC before earnings" -> ['AMC', 'GME']
REVISAR | "I love my Tesla and some berkshire hathaway" -> ['BRK', 'TSLA']
OK  | "DTE is 0 today, pure theta gang" -> []
OK  | "$DTE looks cheap" -> ['DTE']
OK  | "the star of the show" -> []
5 de 6


In [27]:
# Celda M5 - Procesar un archivo (la unidad de trabajo del plan, paso 2)
def procesar_archivo(ruta, anio, mes, tipo):
    """Lee un .zst o .ndjson y devuelve (matriz_mes, metricas)."""
    vig_dia = vigentes_del_mes(anio, mes)
    conteos = defaultdict(Counter)     # fecha -> Counter(ticker)
    tot = rem = leidos = 0
    t0 = time.time()

    def procesa_linea(linea):
        nonlocal tot, rem, leidos
        leidos += 1
        try:
            msg = json.loads(linea)
        except json.JSONDecodeError:
            return
        cu = pd.to_numeric(msg.get('created_utc'), errors='coerce')
        if pd.isna(cu):
            return
        f = pd.Timestamp(int(cu), unit='s').date()
        if f.year != anio or f.month != mes:
            return
        tot += 1
        texto = msg.get('title') if tipo == 'submissions' else msg.get('body')
        if not texto or texto in ('[removed]', '[deleted]'):
            rem += 1
            return
        menciones = detectar(texto) & vig_dia[f]
        for t in menciones:
            conteos[f][t] += 1          # max 1 por mensaje ya garantizado por el set

    if ruta.suffix == '.zst':
        dctx = zstd.ZstdDecompressor(max_window_size=2**31)
        with open(ruta, 'rb') as fh, dctx.stream_reader(fh) as sr:
            for linea in io.TextIOWrapper(sr, encoding='utf-8', errors='ignore'):
                procesa_linea(linea)
    else:
        with open(ruta, encoding='utf-8', errors='ignore') as fh:
            for linea in fh:
                procesa_linea(linea)

    dias = sorted(vig_dia.keys())
    matriz = pd.DataFrame(0, index=pd.Index(dias, name='fecha'),
                          columns=sorted({t for c in conteos.values() for t in c}))
    for f, c in conteos.items():
        for t, n in c.items():
            matriz.loc[f, t] = n
    met = {'leidos': leidos, 'en_mes': tot, 'removidos': rem,
           'con_mencion': int(matriz.values.sum() > 0 and (matriz.sum(axis=1) > 0).sum()),
           'menciones_totales': int(matriz.values.sum()),
           'tickers': matriz.shape[1], 'segundos': round(time.time() - t0, 1)}
    return matriz, met

print('funcion lista')

funcion lista


In [51]:
# Celda M6 - PILOTO: enero de 2020 completo (todos los subs, submissions y comments)
ANIO, MES = 2020, 1
carpeta = DATA / f'{ANIO}' / f'{MES:02d}'
salida = MATRIX / 'mensuales' / f'{ANIO}' / f'{MES:02d}'
salida.mkdir(parents=True, exist_ok=True)

log = []
archivos = sorted(carpeta.glob('*.zst'))
print(len(archivos), 'archivos por procesar')

for ruta in archivos:
    if ruta.stat().st_size == 0:
        print('SALTADO (0 bytes):', ruta.name); continue
    sub = ruta.stem.replace('_submissions', '').replace('_comments', '')
    tipo = 'submissions' if 'submissions' in ruta.name else 'comments'
    matriz, met = procesar_archivo(ruta, ANIO, MES, tipo)
    nombre = f'{ANIO}-{MES:02d}_{sub}_{tipo}.csv'
    matriz.to_csv(salida / nombre)
    met.update({'archivo': ruta.name, 'sub': sub, 'tipo': tipo})
    log.append(met)
    print(f'{nombre:45} msgs {met["en_mes"]:>7} | removidos {met["removidos"]:>6} '
          f'| menciones {met["menciones_totales"]:>7} | tickers {met["tickers"]:>4} | {met["segundos"]}s')

pd.DataFrame(log).to_csv(MATRIX / 'logs' / f'log_{ANIO}-{MES:02d}.csv', index=False)
print('\npiloto guardado en', salida)

16 archivos por procesar
2020-01_Daytrading_comments.csv               msgs    3871 | removidos    337 | menciones     418 | tickers  183 | 0.1s
2020-01_Daytrading_submissions.csv            msgs     482 | removidos      0 | menciones      46 | tickers   31 | 0.0s
2020-01_SecurityAnalysis_comments.csv         msgs    1911 | removidos    251 | menciones     465 | tickers  212 | 0.1s
2020-01_SecurityAnalysis_submissions.csv      msgs     220 | removidos      0 | menciones      68 | tickers   55 | 0.0s
2020-01_StockMarket_comments.csv              msgs    8700 | removidos    562 | menciones    2267 | tickers  566 | 0.2s
2020-01_StockMarket_submissions.csv           msgs     816 | removidos      0 | menciones     208 | tickers  113 | 0.0s
2020-01_investing_comments.csv                msgs   54265 | removidos   4630 | menciones   11026 | tickers 1161 | 1.1s
2020-01_investing_submissions.csv             msgs    3105 | removidos      0 | menciones     471 | tickers  216 | 0.1s
2020-01_options

In [52]:
# Celda M7 - Control de calidad del piloto
import glob
piezas = []
for f in (MATRIX / 'mensuales/2020/01').glob('*.csv'):
    m = pd.read_csv(f, index_col='fecha')
    piezas.append(m)
global_ene = piezas[0].reindex(columns=sorted(set().union(*[p.columns for p in piezas])), fill_value=0)
for p in piezas[1:]:
    global_ene = global_ene.add(p.reindex(columns=global_ene.columns, fill_value=0), fill_value=0)

print('matriz global de enero 2020:', global_ene.shape)
print('\nTOP 25 tickers del mes (REVISION MANUAL DE FALSOS POSITIVOS):')
print(global_ene.sum().sort_values(ascending=False).head(25).to_string())
# Esperado razonable para ene-2020: TSLA arriba (su primer rally), AAPL, AMD, MSFT,
# SPCE subiendo, BA solo si via cashtag. Si ves algo raro (palabras, jerga), lo
# agregamos a exclusiones AHORA, antes de congelar y correr 2020 completo

matriz global de enero 2020: (31, 2385)

TOP 25 tickers del mes (REVISION MANUAL DE FALSOS POSITIVOS):
TSLA    20069
AAPL     9704
AMD      7976
MSFT     5291
SPCE     5202
BYND     3373
BA       2757
NFLX     2329
TWTR     2287
AMZN     2121
INTC     1467
BABA     1027
SBUX     1008
DIS       979
NIO       967
TSM       871
LMT       809
PTON      802
TGT       783
ACB       736
DEAC      690
NVDA      688
WMT       605
BAC       547
ROKU      536


In [53]:
# Celda M7b - Auditoria de rutas de deteccion del top 60 (todas las rutas en una sola pasada)
top60 = global_ene.sum().sort_values(ascending=False).head(60)
kw_por_ticker = defaultdict(list)
for w, s in kw1_de.items():
    for t in s:
        kw_por_ticker[t].append(w)
fr_por_ticker = defaultdict(list)
for lst in kwfrase.values():
    for f, t in lst:
        fr_por_ticker[t].append(f)

print(f'{"ticker":8} {"menciones":>9}  rutas de deteccion')
for t, n in top60.items():
    rutas = []
    if t in token_de.values() or (t.upper() in token_de):
        rutas.append('token')
    if kw_por_ticker.get(t):
        rutas.append('kw:' + ','.join(kw_por_ticker[t]))
    if fr_por_ticker.get(t):
        rutas.append('frase:' + fr_por_ticker[t][0])
    print(f'{t:8} {int(n):>9}  {"; ".join(rutas) if rutas else "solo cashtag"}')
# REVISION: cualquier renglon cuya ruta kw: o frase: sea una palabra comun del
# ingles es candidato a exclusion. Los que solo digan token/cashtag son confiables.

ticker   menciones  rutas de deteccion
TSLA         20069  token; kw:tesla
AAPL          9704  token; kw:apple
AMD           7976  token; frase:advanced micro
MSFT          5291  token; kw:microsoft
SPCE          5202  token; frase:virgin galactic
BYND          3373  token; frase:beyond meat
BA            2757  kw:boeing
NFLX          2329  token; kw:netflix
TWTR          2287  token; kw:twitter
AMZN          2121  token; frase:amazon com
INTC          1467  token; kw:intel
BABA          1027  token; kw:alibaba
SBUX          1008  token; kw:starbucks
DIS            979  token; frase:disney walt
NIO            967  token
TSM            871  token; frase:taiwan semiconductor
LMT            809  token; frase:lockheed martin
PTON           802  token; frase:peloton interactive
TGT            783  token
ACB            736  token; frase:aurora cannabis
DEAC           690  token; frase:diamond eagle
NVDA           688  token; kw:nvidia
WMT            605  token; kw:walmart
BAC            547 

In [54]:
# Celda M8 - Correr un anio completo (con las listas congeladas v9)
ANIO_CORRIDA = 2020

log_anio = []
for mes in range(1, 13):
    carpeta = DATA / f'{ANIO_CORRIDA}' / f'{mes:02d}'
    if not carpeta.exists():
        print(f'{ANIO_CORRIDA}-{mes:02d}: sin carpeta, saltado'); continue
    salida = MATRIX / 'mensuales' / f'{ANIO_CORRIDA}' / f'{mes:02d}'
    salida.mkdir(parents=True, exist_ok=True)
    for ruta in sorted(carpeta.glob('*.zst')) + sorted(carpeta.glob('*.ndjson')):
        sub = ruta.stem.replace('_submissions', '').replace('_comments', '')
        tipo = 'submissions' if 'submissions' in ruta.name else 'comments'
        destino = salida / f'{ANIO_CORRIDA}-{mes:02d}_{sub}_{tipo}.csv'
        if destino.exists():
            continue                      # checkpoint: reanudable, salta lo ya hecho
        if ruta.stat().st_size == 0:
            print('SALTADO (0 bytes):', ruta.name); continue
        matriz, met = procesar_archivo(ruta, ANIO_CORRIDA, mes, tipo)
        matriz.to_csv(destino)
        met.update({'archivo': ruta.name, 'sub': sub, 'tipo': tipo, 'mes': mes})
        log_anio.append(met)
    print(f'{ANIO_CORRIDA}-{mes:02d} listo')

pd.DataFrame(log_anio).to_csv(MATRIX / 'logs' / f'log_{ANIO_CORRIDA}.csv', index=False)
print('anio completo:', ANIO_CORRIDA, '|', len(log_anio), 'archivos procesados en esta corrida')

2020-01 listo
2020-02 listo
2020-03 listo
2020-04 listo
2020-05 listo
2020-06 listo
2020-07 listo
2020-08 listo
2020-09 listo
2020-10 listo
2020-11 listo
2020-12 listo
anio completo: 2020 | 280 archivos procesados en esta corrida


In [ ]:
# Celda M8 - Correr un anio completo (con las listas congeladas v9)
ANIO_CORRIDA = 2021
# Nota resuelta: los ceros de submissions de Daytrading (ene-feb 2021) son un hueco real del dump, verificado en la celda 0.1 de matrices_paso0
log_anio = []
for mes in range(1, 13):
    carpeta = DATA / f'{ANIO_CORRIDA}' / f'{mes:02d}'
    if not carpeta.exists():
        print(f'{ANIO_CORRIDA}-{mes:02d}: sin carpeta, saltado'); continue
    salida = MATRIX / 'mensuales' / f'{ANIO_CORRIDA}' / f'{mes:02d}'
    salida.mkdir(parents=True, exist_ok=True)
    for ruta in sorted(carpeta.glob('*.zst')) + sorted(carpeta.glob('*.ndjson')):
        sub = ruta.stem.replace('_submissions', '').replace('_comments', '')
        tipo = 'submissions' if 'submissions' in ruta.name else 'comments'
        destino = salida / f'{ANIO_CORRIDA}-{mes:02d}_{sub}_{tipo}.csv'
        if destino.exists():
            continue                      # checkpoint: reanudable, salta lo ya hecho
        if ruta.stat().st_size == 0:
            print('SALTADO (0 bytes):', ruta.name); continue
        matriz, met = procesar_archivo(ruta, ANIO_CORRIDA, mes, tipo)
        matriz.to_csv(destino)
        met.update({'archivo': ruta.name, 'sub': sub, 'tipo': tipo, 'mes': mes})
        log_anio.append(met)
    print(f'{ANIO_CORRIDA}-{mes:02d} listo')

pd.DataFrame(log_anio).to_csv(MATRIX / 'logs' / f'log_{ANIO_CORRIDA}.csv', index=False)
print('anio completo:', ANIO_CORRIDA, '|', len(log_anio), 'archivos procesados en esta corrida')

In [ ]:
# Celda M8 - Correr un anio completo (con las listas congeladas v9)
ANIO_CORRIDA = 2022

log_anio = []
for mes in range(1, 13):
    carpeta = DATA / f'{ANIO_CORRIDA}' / f'{mes:02d}'
    if not carpeta.exists():
        print(f'{ANIO_CORRIDA}-{mes:02d}: sin carpeta, saltado'); continue
    salida = MATRIX / 'mensuales' / f'{ANIO_CORRIDA}' / f'{mes:02d}'
    salida.mkdir(parents=True, exist_ok=True)
    for ruta in sorted(carpeta.glob('*.zst')) + sorted(carpeta.glob('*.ndjson')):
        sub = ruta.stem.replace('_submissions', '').replace('_comments', '')
        tipo = 'submissions' if 'submissions' in ruta.name else 'comments'
        destino = salida / f'{ANIO_CORRIDA}-{mes:02d}_{sub}_{tipo}.csv'
        if destino.exists():
            continue                      # checkpoint: reanudable, salta lo ya hecho
        if ruta.stat().st_size == 0:
            print('SALTADO (0 bytes):', ruta.name); continue
        matriz, met = procesar_archivo(ruta, ANIO_CORRIDA, mes, tipo)
        matriz.to_csv(destino)
        met.update({'archivo': ruta.name, 'sub': sub, 'tipo': tipo, 'mes': mes})
        log_anio.append(met)
    print(f'{ANIO_CORRIDA}-{mes:02d} listo')

pd.DataFrame(log_anio).to_csv(MATRIX / 'logs' / f'log_{ANIO_CORRIDA}.csv', index=False)
print('anio completo:', ANIO_CORRIDA, '|', len(log_anio), 'archivos procesados en esta corrida')

In [ ]:
# Celda M8 - Correr un anio completo (con las listas congeladas v9)
ANIO_CORRIDA = 2023

log_anio = []
for mes in range(1, 13):
    carpeta = DATA / f'{ANIO_CORRIDA}' / f'{mes:02d}'
    if not carpeta.exists():
        print(f'{ANIO_CORRIDA}-{mes:02d}: sin carpeta, saltado'); continue
    salida = MATRIX / 'mensuales' / f'{ANIO_CORRIDA}' / f'{mes:02d}'
    salida.mkdir(parents=True, exist_ok=True)
    for ruta in sorted(carpeta.glob('*.zst')) + sorted(carpeta.glob('*.ndjson')):
        sub = ruta.stem.replace('_submissions', '').replace('_comments', '')
        tipo = 'submissions' if 'submissions' in ruta.name else 'comments'
        destino = salida / f'{ANIO_CORRIDA}-{mes:02d}_{sub}_{tipo}.csv'
        if destino.exists():
            continue                      # checkpoint: reanudable, salta lo ya hecho
        if ruta.stat().st_size == 0:
            print('SALTADO (0 bytes):', ruta.name); continue
        matriz, met = procesar_archivo(ruta, ANIO_CORRIDA, mes, tipo)
        matriz.to_csv(destino)
        met.update({'archivo': ruta.name, 'sub': sub, 'tipo': tipo, 'mes': mes})
        log_anio.append(met)
    print(f'{ANIO_CORRIDA}-{mes:02d} listo')

pd.DataFrame(log_anio).to_csv(MATRIX / 'logs' / f'log_{ANIO_CORRIDA}.csv', index=False)
print('anio completo:', ANIO_CORRIDA, '|', len(log_anio), 'archivos procesados en esta corrida')

In [ ]:
# Celda M8 - Correr un anio completo (con las listas congeladas v9)
ANIO_CORRIDA = 2024

log_anio = []
for mes in range(1, 13):
    carpeta = DATA / f'{ANIO_CORRIDA}' / f'{mes:02d}'
    if not carpeta.exists():
        print(f'{ANIO_CORRIDA}-{mes:02d}: sin carpeta, saltado'); continue
    salida = MATRIX / 'mensuales' / f'{ANIO_CORRIDA}' / f'{mes:02d}'
    salida.mkdir(parents=True, exist_ok=True)
    for ruta in sorted(carpeta.glob('*.zst')) + sorted(carpeta.glob('*.ndjson')):
        sub = ruta.stem.replace('_submissions', '').replace('_comments', '')
        tipo = 'submissions' if 'submissions' in ruta.name else 'comments'
        destino = salida / f'{ANIO_CORRIDA}-{mes:02d}_{sub}_{tipo}.csv'
        if destino.exists():
            continue                      # checkpoint: reanudable, salta lo ya hecho
        if ruta.stat().st_size == 0:
            print('SALTADO (0 bytes):', ruta.name); continue
        matriz, met = procesar_archivo(ruta, ANIO_CORRIDA, mes, tipo)
        matriz.to_csv(destino)
        met.update({'archivo': ruta.name, 'sub': sub, 'tipo': tipo, 'mes': mes})
        log_anio.append(met)
    print(f'{ANIO_CORRIDA}-{mes:02d} listo')

pd.DataFrame(log_anio).to_csv(MATRIX / 'logs' / f'log_{ANIO_CORRIDA}.csv', index=False)
print('anio completo:', ANIO_CORRIDA, '|', len(log_anio), 'archivos procesados en esta corrida')

In [ ]:
# Celda M8 - Correr un anio completo (con las listas congeladas v9)
ANIO_CORRIDA = 2025

log_anio = []
for mes in range(1, 13):
    carpeta = DATA / f'{ANIO_CORRIDA}' / f'{mes:02d}'
    if not carpeta.exists():
        print(f'{ANIO_CORRIDA}-{mes:02d}: sin carpeta, saltado'); continue
    salida = MATRIX / 'mensuales' / f'{ANIO_CORRIDA}' / f'{mes:02d}'
    salida.mkdir(parents=True, exist_ok=True)
    for ruta in sorted(carpeta.glob('*.zst')) + sorted(carpeta.glob('*.ndjson')):
        sub = ruta.stem.replace('_submissions', '').replace('_comments', '')
        tipo = 'submissions' if 'submissions' in ruta.name else 'comments'
        destino = salida / f'{ANIO_CORRIDA}-{mes:02d}_{sub}_{tipo}.csv'
        if destino.exists():
            continue                      # checkpoint: reanudable, salta lo ya hecho
        if ruta.stat().st_size == 0:
            print('SALTADO (0 bytes):', ruta.name); continue
        matriz, met = procesar_archivo(ruta, ANIO_CORRIDA, mes, tipo)
        matriz.to_csv(destino)
        met.update({'archivo': ruta.name, 'sub': sub, 'tipo': tipo, 'mes': mes})
        log_anio.append(met)
    print(f'{ANIO_CORRIDA}-{mes:02d} listo')

pd.DataFrame(log_anio).to_csv(MATRIX / 'logs' / f'log_{ANIO_CORRIDA}.csv', index=False)
print('anio completo:', ANIO_CORRIDA, '|', len(log_anio), 'archivos procesados en esta corrida')

In [62]:
# Celda M8 - Correr un anio completo (con las listas congeladas v9)
ANIO_CORRIDA = 2026

log_anio = []
for mes in range(1, 13):
    carpeta = DATA / f'{ANIO_CORRIDA}' / f'{mes:02d}'
    if not carpeta.exists():
        print(f'{ANIO_CORRIDA}-{mes:02d}: sin carpeta, saltado'); continue
    salida = MATRIX / 'mensuales' / f'{ANIO_CORRIDA}' / f'{mes:02d}'
    salida.mkdir(parents=True, exist_ok=True)
    for ruta in sorted(carpeta.glob('*.zst')) + sorted(carpeta.glob('*.ndjson')):
        sub = ruta.stem.replace('_submissions', '').replace('_comments', '')
        tipo = 'submissions' if 'submissions' in ruta.name else 'comments'
        destino = salida / f'{ANIO_CORRIDA}-{mes:02d}_{sub}_{tipo}.csv'
        if destino.exists():
            continue                      # checkpoint: reanudable, salta lo ya hecho
        if ruta.stat().st_size == 0:
            print('SALTADO (0 bytes):', ruta.name); continue
        matriz, met = procesar_archivo(ruta, ANIO_CORRIDA, mes, tipo)
        matriz.to_csv(destino)
        met.update({'archivo': ruta.name, 'sub': sub, 'tipo': tipo, 'mes': mes})
        log_anio.append(met)
    print(f'{ANIO_CORRIDA}-{mes:02d} listo')

pd.DataFrame(log_anio).to_csv(MATRIX / 'logs' / f'log_{ANIO_CORRIDA}.csv', index=False)
print('anio completo:', ANIO_CORRIDA, '|', len(log_anio), 'archivos procesados en esta corrida')

2026-01 listo
2026-02 listo
2026-03 listo
2026-04 listo
2026-05 listo
2026-06 listo
2026-07: sin carpeta, saltado
2026-08: sin carpeta, saltado
2026-09: sin carpeta, saltado
2026-10: sin carpeta, saltado
2026-11: sin carpeta, saltado
2026-12: sin carpeta, saltado
anio completo: 2026 | 192 archivos procesados en esta corrida


In [67]:
# Celda de limpieza - borrar matrices y logs para regenerar con v11
import shutil
shutil.rmtree(MATRIX / 'mensuales')
for lg in (MATRIX / 'logs').glob('log_*.csv'):
    lg.unlink()
(MATRIX / 'mensuales').mkdir()
print('listo para regenerar')

listo para regenerar


In [68]:
# Celda M9 - corrida total 2020-2026 (checkpoint por archivo, reanudable)
import time

for ANIO_CORRIDA in range(2020, 2027):
    t0 = time.time()
    log_anio = []
    for mes in range(1, 13):
        carpeta = DATA / f'{ANIO_CORRIDA}' / f'{mes:02d}'
        if not carpeta.exists():
            continue  # p.ej. 2026-07 en adelante
        salida = MATRIX / 'mensuales' / f'{ANIO_CORRIDA}' / f'{mes:02d}'
        salida.mkdir(parents=True, exist_ok=True)
        archivos = sorted(carpeta.glob('*.zst')) + sorted(carpeta.glob('*.ndjson'))
        for ruta in archivos:
            sub, tipo = ruta.stem.rsplit('_', 1)
            destino = salida / f'{ANIO_CORRIDA}-{mes:02d}_{sub}_{tipo}.csv'
            if destino.exists():
                continue  # checkpoint: ya procesado
            if ruta.stat().st_size == 0:
                print(f'SALTADO (0 bytes): {ruta.name}')
                continue
            matriz, met = procesar_archivo(ruta, ANIO_CORRIDA, mes, tipo)
            matriz.to_csv(destino)
            met['archivo'] = ruta.name
            met['sub'] = sub
            met['tipo'] = tipo
            log_anio.append(met)
        print(f'{ANIO_CORRIDA}-{mes:02d} listo', flush=True)
    if log_anio:
        pd.DataFrame(log_anio).to_csv(MATRIX / 'logs' / f'log_{ANIO_CORRIDA}.csv', index=False)
    print(f'anio completo: {ANIO_CORRIDA} | {len(log_anio)} archivos | {(time.time()-t0)/60:.1f} min')

print('CORRIDA TOTAL 2020-2026 TERMINADA')

2020-01 listo
2020-02 listo
2020-03 listo
2020-04 listo
2020-05 listo
2020-06 listo
2020-07 listo
2020-08 listo
2020-09 listo
2020-10 listo
2020-11 listo
2020-12 listo
anio completo: 2020 | 280 archivos | 5.9 min
2021-01 listo
2021-02 listo
2021-03 listo
2021-04 listo
2021-05 listo
2021-06 listo
2021-07 listo
2021-08 listo
2021-09 listo
2021-10 listo
2021-11 listo
2021-12 listo
anio completo: 2021 | 366 archivos | 14.9 min
2022-01 listo
2022-02 listo
2022-03 listo
2022-04 listo
2022-05 listo
2022-06 listo
2022-07 listo
2022-08 listo
2022-09 listo
2022-10 listo
2022-11 listo
2022-12 listo
anio completo: 2022 | 384 archivos | 9.0 min
2023-01 listo
2023-02 listo
2023-03 listo
2023-04 listo
2023-05 listo
2023-06 listo
2023-07 listo
2023-08 listo
2023-09 listo
2023-10 listo
2023-11 listo
2023-12 listo
anio completo: 2023 | 382 archivos | 4.6 min
2024-01 listo
2024-02 listo
2024-03 listo
2024-04 listo
2024-05 listo
2024-06 listo
2024-07 listo
2024-08 listo
2024-09 listo
2024-10 listo
2024-11

In [69]:
# Celda M10 - Matrices maestras: submissions y comments (2020-01-01 a 2026-06-30)
import time

t0 = time.time()
D0, D1 = pd.Timestamp('2020-01-01'), pd.Timestamp('2026-06-30')
dias_totales = pd.date_range(D0, D1, freq='D')

maestras = {}
for tipo in ('submissions', 'comments'):
    piezas = []
    for anio in range(2020, 2027):
        for mes in range(1, 13):
            carpeta = MATRIX / 'mensuales' / f'{anio}' / f'{mes:02d}'
            if not carpeta.exists():
                continue
            rutas = sorted(carpeta.glob(f'*_{tipo}.csv'))
            if not rutas:
                continue
            acum = None
            for ruta in rutas:
                df = pd.read_csv(ruta, index_col=0, keep_default_na=False, na_values=[''])
                df.index = pd.to_datetime(df.index)
                acum = df if acum is None else acum.add(df, fill_value=0)
            piezas.append(acum)
        print(f'{tipo} {anio} listo', flush=True)
    maestra = pd.concat(piezas).fillna(0)
    maestra = maestra.groupby(maestra.index).sum()   # blindaje por si un dia quedara duplicado
    maestra = maestra.reindex(dias_totales, fill_value=0).astype(int)
    maestra = maestra[sorted(maestra.columns)]
    maestra.index.name = 'fecha'
    maestras[tipo] = maestra
    maestra.to_csv(MATRIX / 'maestras' / f'maestra_{tipo}_2020_2026.csv')
    print(f'MAESTRA {tipo}: {maestra.shape[0]} dias x {maestra.shape[1]} tickers | '
          f'total {int(maestra.values.sum()):,}')

print(f'terminado en {(time.time()-t0)/60:.1f} min')

submissions 2020 listo
submissions 2021 listo
submissions 2022 listo
submissions 2023 listo
submissions 2024 listo
submissions 2025 listo
submissions 2026 listo
MAESTRA submissions: 2373 dias x 7919 tickers | total 1,965,100
comments 2020 listo
comments 2021 listo
comments 2022 listo
comments 2023 listo
comments 2024 listo
comments 2025 listo
comments 2026 listo
MAESTRA comments: 2373 dias x 8792 tickers | total 25,843,758
terminado en 1.2 min


In [70]:
# Celda M11 - Matriz de matrices (submissions + comments) y checks
sub_m = maestras['submissions']
com_m = maestras['comments']

global_m = sub_m.add(com_m, fill_value=0).astype(int)
global_m = global_m[sorted(global_m.columns)]
global_m.index.name = 'fecha'
global_m.to_csv(MATRIX / 'maestras' / 'matriz_global_2020_2026.csv')

print(f'GLOBAL: {global_m.shape[0]} dias x {global_m.shape[1]} tickers | '
      f'total {int(global_m.values.sum()):,}')

# check 1: conservacion de la suma
assert int(global_m.values.sum()) == int(sub_m.values.sum()) + int(com_m.values.sum())

# check 2: ancla GME
dia = pd.Timestamp('2021-01-28')
g_s = int(sub_m.at[dia, 'GME']) if 'GME' in sub_m.columns else 0
g_c = int(com_m.at[dia, 'GME']) if 'GME' in com_m.columns else 0
print(f'GME 28-ene-2021: {g_s:,} (posts) + {g_c:,} (comments) = {int(global_m.at[dia, "GME"]):,}')
assert g_s + g_c == int(global_m.at[dia, 'GME'])

# check 3: dias sin ninguna mencion (no deberia haber, o muy pocos al inicio)
dias_cero = global_m.sum(axis=1)
print('dias con cero menciones:', int((dias_cero == 0).sum()))

print('top 10 historico:', {t: f'{v:,}' for t, v in global_m.sum().nlargest(10).items()})

GLOBAL: 2373 dias x 8812 tickers | total 27,808,858
GME 28-ene-2021: 33,524 (posts) + 108,433 (comments) = 141,957
dias con cero menciones: 0
top 10 historico: {'GME': '4,355,809', 'TSLA': '1,770,101', 'AMC': '1,091,123', 'AAPL': '813,655', 'NVDA': '784,753', 'RDDT': '723,325', 'TWTR': '468,064', 'MSFT': '406,273', 'PLTR': '405,358', 'AMD': '345,581'}


In [71]:
# Celda M12 - Maestras por subreddit (posts+comments; robustez de deteccion por sub)
import collections

(MATRIX / 'maestras' / 'por_subreddit').mkdir(parents=True, exist_ok=True)

rutas_por_sub = collections.defaultdict(list)
for ruta in (MATRIX / 'mensuales').glob('*/*/*.csv'):
    sub = ruta.stem.split('_', 1)[1].rsplit('_', 1)[0]   # 2020-01_Sub_tipo -> Sub
    rutas_por_sub[sub].append(ruta)

suma_subs = 0
for sub in sorted(rutas_por_sub):
    piezas = []
    for ruta in sorted(rutas_por_sub[sub]):
        df = pd.read_csv(ruta, index_col=0, keep_default_na=False, na_values=[''])
        df.index = pd.to_datetime(df.index)
        piezas.append(df)
    m = pd.concat(piezas).fillna(0)
    m = m.groupby(m.index).sum()      # submissions y comments comparten fechas: aqui se suman
    m = m.reindex(dias_totales, fill_value=0).astype(int)
    m = m[sorted(m.columns)]
    m.index.name = 'fecha'
    m.to_csv(MATRIX / 'maestras' / 'por_subreddit' / f'maestra_{sub}_2020_2026.csv')
    suma_subs += int(m.values.sum())
    print(f'{sub}: {m.shape[1]} tickers | total {int(m.values.sum()):,}')

# check de conservacion: la suma de los 16 subreddits debe igualar la matriz global
print('suma de los 16 subs:', f'{suma_subs:,}', '| global:', f'{int(global_m.values.sum()):,}')
assert suma_subs == int(global_m.values.sum())
print('CONSERVACION OK: los 16 subreddits suman exactamente la matriz global')

Daytrading: 5333 tickers | total 254,210
SPACs: 4691 tickers | total 546,288
SatoshiStreetBets: 2192 tickers | total 122,580
SecurityAnalysis: 2258 tickers | total 21,397
Shortsqueeze: 5323 tickers | total 536,922
SqueezePlays: 2832 tickers | total 47,197
StockMarket: 6223 tickers | total 504,691
Superstonk: 5630 tickers | total 4,496,447
Vitards: 3700 tickers | total 311,054
WallStreetbetsELITE: 5422 tickers | total 699,469
Wallstreetbetsnew: 4658 tickers | total 366,088
investing: 6528 tickers | total 984,866
options: 4605 tickers | total 343,408
pennystocks: 6902 tickers | total 953,993
stocks: 7452 tickers | total 2,382,859
wallstreetbets: 8196 tickers | total 15,237,389
suma de los 16 subs: 27,808,858 | global: 27,808,858
CONSERVACION OK: los 16 subreddits suman exactamente la matriz global
